In [ ]:
# カレントディレクトリを変更
import os
os.chdir('../')

# 必要なライブラリをインストール
# ノートブック環境では先頭に ! をつけてシェルコマンドとして実行します
!pip install snowflake-snowpark-python==1.39.0
!pip install -r requirements.txt

In [ ]:
# python再起動
%restart_python

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import glob
import importlib.util
import shutil
from snowflake.snowpark import Session

BASE_DIR = '/Workspace/Users/1640138@tuc.tbfr.toyota.co.jp/congestion_and_sporty_detection'  # + 親ディレクトリ

DOTENV_ABS = f"{BASE_DIR}/.env"
TOKEN_PICKLE_ABS = f"{BASE_DIR}/token.pickle"
load_dotenv(DOTENV_ABS, override=True)    # envファイルの読み込み、同名の変数があれば上書き。
os.environ['OAUTH_TOKEN_FILE_PATH'] = TOKEN_PICKLE_ABS  # ★ Import より前に設定
MODULE_PATH = os.path.join(BASE_DIR, 'snowflake_oauth.py')
spec = importlib.util.spec_from_file_location('snowflake_oauth', MODULE_PATH)
snowflake_oauth = importlib.util.module_from_spec(spec)
spec.loader.exec_module(snowflake_oauth)

print("CWD:", os.getcwd())

os.chdir('../')
from snowflake_oauth import fetch_token, access_token

t = fetch_token()
print("token keys:", list(t.keys()))
print("access_token len:", len(access_token()))

sf_account = os.getenv("SNOWFLAKE_ACCOUNT")
sf_user = os.getenv("SNOWFLAKE_USER")
sf_database = os.getenv("SNOWFLAKE_DB")
missing = [k for k,v in {"SNOWFLAKE_ACCOUNT": sf_account, "SNOWFLAKE_USER": sf_user, "SNOWFLAKE_DB": sf_database}.items() if not v]
if missing:
    raise RuntimeError(f"必須環境変数が未設定です: {missing}")
params = {
    "account":    sf_account,
    "user":       sf_user,
    "database":   sf_database,
    "token":      access_token(),  # 文字列
    "authenticator": "oauth",
}
session = Session.builder.configs(params).create()
print("✅ Snowpark session ready")

In [ ]:
# --- handover/ を import パスへ (notebooks/ の 1つ上) ---
import sys, os, importlib
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parents[1]           # notebooks/ + handover/
sys.path.insert(0, str(PROJECT_ROOT))          # ★ src の"殻だけ"を追加 (src 自体は追加しない)

# --- .env を読み込む (上書きしない) ---
load_dotenv(PROJECT_ROOT / '.env', override=False)

# --- トークンファイルの場所 (OAUTH_TOKEN_FILE_PATH="token_pickle") ---
os.environ["OAUTH_TOKEN_FILE_PATH"] = str(PROJECT_ROOT / "token_pickle")

# --- handover 直下の snowflake_oauth を通常 import ---
_sfo = importlib.import_module("snowflake_oauth")    # handover/snowflake_oauth.py

# --- 既存コード互換: 'from src import snowflake_oauth' でも解決できるように別名登録 ---
import src as _src                                      # handover/src/__init__.py (空でOK)
setattr(_src, "snowflake_oauth", _sfo)

print("✅ bootstrap (.env only) done")

データの読み込み

In [ ]:
from src import query_utils as q

df1 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": "VAP_SOH_ANALYTIX_1640138_TTTC_READONLY", "table_name": "VERITAS_CAN_20260306105528"}
)

df2 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": "VAP_SOH_ANALYTIX_1640138_TTTC_READONLY", "table_name": "VERITAS_CAN_20260306110159"}
)

df3 = q.get_snowflake_table(
    snowflake_session=session,
    data_params={"schema_name": "VAP_SOH_ANALYTIX_1640138_TTTC_READONLY", "table_name": "VERITAS_CAN_20260306110603"}
)

# 例：不要列を削除（複数可）
drop_cols = ['VALUE', 'LINK_ID', 'SP1', 'ACC_BOB', 'LATITUDE', 'LONGITUDE',
             'NMLATITUDE', 'NMLONGITUDE', 'PT_DT']

crown_2411_df = df1.drop(*drop_cols)
crown_2506_df = df2.drop(*drop_cols)
crown_2601_df = df3.drop(*drop_cols)

In [ ]:
import pandas as pd
import numpy as np
import json
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, DoubleType

# 1)~8)のロジックを内包する処理関数
def process_single_trip(trip_df: pd.DataFrame) -> pd.DataFrame:
    # --- 初期設定・データ不足時のデフォルト返り値 ---
    vin = str(trip_df['MASKED_VIN'].iloc[0]) if not trip_df.empty else None
    trip_id = int(trip_df['TRIPCOUNT'].iloc[0]) if not trip_df.empty else None
    empty_result = pd.DataFrame([{
        'MASKED_VIN': vin,
        'TRIPCOUNT': trip_id,
        'LABEL': '保留(データ不足)',
        'N_EVENTS': 0,
        'T_MINUTES': 0.0,
        'RATE': 0.0,
        'EVENTS_JSON': '[]'
    }])

    # 31サンプル(約31秒)未満はバイアス計算ができないため弾く
    if len(trip_df) < 31:
        return empty_result
    
    # タイムスタンプでソート（時系列処理に必須）
    trip_df['GPS_TIMESTAMP'] = pd.to_datetime(trip_df['GPS_TIMESTAMP'])
    trip_df = trip_df.sort_values('GPS_TIMESTAMP').reset_index(drop=True)

    # 2) 閾値設定
    GRAVITY = 9.80665
    HIGH_G = 0.40
    LOW_G = 0.35

    # 3) 単位正規化・品質チェック（異常値と欠損値の処理）
    # 要件に合わせ 1.2Gを外れ値の閾値とする
    limit_accel = 1.2
    trip_df.loc[trip_df['ACCELERATIONFB'].abs() >= limit_accel, 'ACCELERATIONFB'] = np.nan
    trip_df.loc[trip_df['ACCELERATIONLR'].abs() >= limit_accel, 'ACCELERATIONLR'] = np.nan
    # 1サンプル以内の欠損を線形補間し、補間しきれなかった行(連続欠損)は削除
    trip_df['ACCELERATIONFB'] = trip_df['ACCELERATIONFB'].interpolate(limit=1)
    trip_df['ACCELERATIONLR'] = trip_df['ACCELERATIONLR'].interpolate(limit=1)
    trip_df = trip_df.dropna(subset=['ACCELERATIONFB', 'ACCELERATIONLR']).reset_index(drop=True)
    # 欠損削除後に再びデータ数チェック
    if len(trip_df) < 31:
        return empty_result
    
    # 4) バイアス補正・軽量平滑化
    # 3点メディアン（平滑化）
    ax_smooth = trip_df['ACCELERATIONFB'].rolling(window=3, center=True, min_periods=1).median()
    ay_smooth = trip_df['ACCELERATIONLR'].rolling(window=3, center=True, min_periods=1).median()
    # 31点メディアン（DCバイアス算出）
    ax_bias = ax_smooth.rolling(window=31, center=True, min_periods=1).median()
    ay_bias = ay_smooth.rolling(window=31, center=True, min_periods=1).median()
    # バイアス除去（補正）
    ax_prime = ax_smooth - ax_bias
    ay_prime = ay_smooth - ay_bias

    # 5) 合成Gの算出（ここで 重力加速度 で割り、G単位に変換）
    trip_df['G_mag'] = np.sqrt(ax_prime**2 + ay_prime**2) / GRAVITY

    # 6) イベント抽出
    events = []
    is_event = False
    start_time = None
    for idx, row in trip_df.iterrows():
        g = row['G_mag']
        t = row['GPS_TIMESTAMP']
        # 開始条件
        if not is_event and g > HIGH_G:
            is_event = True
            start_time = t
        # 終了条件
        elif is_event and g < LOW_G:
            is_event = False
            events.append({'start': start_time, 'end': t})

    # 終了処理: トリップの最後でイベント継続中の場合はその時点を終了とする
    if is_event:
        events.append({'start': start_time, 'end': trip_df['GPS_TIMESTAMP'].iloc[-1]})

    # イベントの統合（終了から次の開始までが2秒以内なら1件にまとめる）
    merged_events = []
    if len(events) > 0:
        current_event = events[0]
        for next_event in events[1:]:
            gap_seconds = (next_event['start'] - current_event['end']).total_seconds()
            if gap_seconds <= 2.0:
                current_event['end'] = next_event['end']  # 終了時間を延長して統合
            else:
                merged_events.append(current_event)
                current_event = next_event
        merged_events.append(current_event)

    # 7) スポーティ運転イベント集計
    N = len(merged_events)
    T_minutes = (trip_df['GPS_TIMESTAMP'].max() - trip_df['GPS_TIMESTAMP'].min()).total_seconds() / 60.0
    R = N / T_minutes if T_minutes > 0 else 0.0

    # 8) スポーティー運転判定
    if T_minutes < 5.0:
        label = '保留(絶対所要不足)'
    else:
        label = 'スポーティ' if R >= 1.5 else '非スポーティ'

    # 9) テーブル出力（各イベントの開始/終了時刻をJSON文字列化して保存）
    # デバッグや後続の可視化(BIツール等)で扱いやすいように整形します
    events_list = [{'start': ev['start'].strftime('%Y-%m-%d %H:%M:%S'),
                    'end':   ev['end'].strftime('%Y-%m-%d %H:%M:%S')} for ev in merged_events]
    events_json = json.dumps(events_list)
    
    # 戻り値は必ず以下のスキーマ定義と一致する 1行のPandas DataFrame を返す
    return pd.DataFrame([{
        'MASKED_VIN': vin,
        'TRIPCOUNT': trip_id,
        'LABEL': label,
        'N_EVENTS': int(N),
        'T_MINUTES': float(T_minutes),
        'RATE': float(R),
        'EVENTS_JSON': events_json
    }])

In [ ]:
# --- 1. カレントデータベースとスキーマの設定 ---
# データベースは環境変数から取得しているものを指定
import os
session.use_database(os.getenv("SNOWFLAKE_DB"))

# 書き込み（一時表の作成）が許可されているスキーマを指定
session.use_schema("VAP_SCH_ANALYTIX_1640138_TTTC")

# --- 2. スキーマの定義 ---
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, DoubleType
result_schema = StructType([
    StructField("MASKED_VIN", StringType()),
    StructField("TRIPCOUNT", IntegerType()),
    StructField("LABEL", StringType()),
    StructField("N_EVENTS", IntegerType()),
    StructField("T_MINUTES", DoubleType()),
    StructField("RATE", DoubleType()),
    StructField("EVENTS_JSON", StringType())
])

# --- 3. 分散処理の実行 ---
result_spark_df = crown_2411_df.groupBy("MASKED_VIN", "TRIPCOUNT").applyInPandas(
    process_single_trip,
    output_schema=result_schema
)

# 結果の表示
result_spark_df.show()